In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

## Fact

### Which feature had the strongest correlation with disease progression?

In [ ]:
corr = df.drop(columns='target').corrwith(df['target']).abs().sort_values(ascending=False)
print(corr.head())
print(f"\nStrongest correlation: {corr.idxmax()} ({corr.max():.3f})")

### Did patients with higher BMI generally show higher disease progression values?

**Method:** Reuse absolute correlation from the previous result, then run `scipy.stats.pearsonr` to get the signed r and p-value.

**Pearson r** measures linear association between BMI and progression on a scale of −1 to 1. A positive r means higher BMI tends to go with higher progression; magnitude indicates strength (|r| > 0.5 is considered moderate-to-strong).

**p-value** is the probability of observing a correlation this large by chance under the null hypothesis (no relationship). p < 0.05 is the conventional significance threshold; values like 1e-40 make the result effectively certain.

**Quartile grouping** splits BMI into four equal-sized bins (Q1 = lowest, Q4 = highest) and computes mean progression per bin — a model-free way to confirm the trend is monotonic, not just driven by outliers.

In [ ]:
from scipy import stats

r, p = stats.pearsonr(df['bmi'], df['target'])
print(f"BMI corr (abs): {corr['bmi']:.3f}  |  Pearson r: {r:.3f}  |  p-value: {p:.2e}")

means = df.groupby(pd.qcut(df['bmi'], 4, labels=['Q1','Q2','Q3','Q4']))['target'].mean().round(1)
print(means)
print(f"\nYes — mean progression rises monotonically from Q1 to Q4 (r={r:.3f}, p={p:.2e}).")

### Was blood pressure (bp) evenly distributed across patients?

In [12]:
stat, p = stats.normaltest(df['bp'])
print(df['bp'].describe().round(3))
print(f"\nSkewness: {df['bp'].skew():.3f}")
print(f"D'Agostino-Pearson: stat={stat:.3f}, p={p:.4f}")
print(f"\nNo — normality rejected (p={p:.4f}); mild right skew indicates uneven distribution.")

count    442.000
mean      -0.000
std        0.048
min       -0.112
25%       -0.037
50%       -0.006
75%        0.036
max        0.132
Name: bp, dtype: float64

Skewness: 0.291
D'Agostino-Pearson: stat=15.621, p=0.0004

No — normality rejected (p=0.0004); mild right skew indicates uneven distribution.


### Which feature showed the greatest variance?

In [15]:
raw = pd.DataFrame(load_diabetes(scaled=False).data, columns=data.feature_names)
var = raw.var().sort_values(ascending=False)
print(var.round(2))
print(f"\nNote: scaled data is meaningless here — all variances equal after standardization.")
print(f"Answer: {var.idxmax()} has the greatest variance ({var.max():.2f}) on the raw scale.")

s1     1197.72
s2      924.96
bp      191.30
age     171.85
s3      167.29
s6      132.17
bmi      19.52
s4        1.67
s5        0.27
sex       0.25
dtype: float64

Note: scaled data is meaningless here — all variances equal after standardization.
Answer: s1 has the greatest variance (1197.72) on the raw scale.


### Were there any obvious outliers in the dataset?

In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
import numpy as np

X, y = df.drop(columns='target'), df['target']

# Regression: sort |residuals|, find first sharp jump
res = np.abs(y - LinearRegression().fit(X, y).predict(X))
res_sorted = np.sort(res)
diffs = np.diff(res_sorted)
t_reg = res_sorted[:-1][diffs > diffs.mean() + 2 * diffs.std()][0]
reg_idx = set(np.where(res > t_reg)[0])

# Isolation Forest: sort anomaly scores, find first sharp jump
scores = -IsolationForest(random_state=0).fit(X).decision_function(X)
sc_sorted = np.sort(scores)
diffs_s = np.diff(sc_sorted)
t_if = sc_sorted[:-1][diffs_s > diffs_s.mean() + 2 * diffs_s.std()][0]
if_idx = set(np.where(scores > t_if)[0])

print(f"Regression  (threshold={t_reg:.1f}):  {len(reg_idx)} outliers")
print(f"Isolation Forest (threshold={t_if:.3f}): {len(if_idx)} outliers")
print(f"\nOverlap: {len(reg_idx & if_idx)} shared | reg-only: {len(reg_idx - if_idx)} | IF-only: {len(if_idx - reg_idx)}")
print(f"\nAnswer: {'Yes' if reg_idx or if_idx else 'No'} — both methods agree on {len(reg_idx & if_idx)} outliers via sharp error growth.")

Regression  (threshold=101.6):  19 outliers
Isolation Forest (threshold=-0.114): 440 outliers

Overlap: 18 shared | reg-only: 1 | IF-only: 422

Answer: Yes — both methods agree on 18 outliers via sharp error growth.


## Verification

### Was BMI truly more important than age in predicting disease progression?

### Did patients with higher blood pressure always experience more severe diabetes progression?

### Were the cholesterol-related features (s1–s6) strongly correlated with each other?

### Was the relationship between age and disease progression strong enough to be captured by a linear model?

### Did feature standardization significantly affect model performance?

### Did using all features necessarily produce better predictions than using only a few important features?

### Did BMI and blood sugar related indicators (such as s5) jointly influence disease progression?

## Exploratory

### If you could keep only one feature to predict disease progression, which would you choose and why?

### Would non-linear models such as Random Forest outperform linear models, and why?